In [ ]:
# ==============================================================================
# Step 1: Import Libraries
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import warnings
import itertools
import pickle
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- Initial Setup ---
warnings.filterwarnings("ignore")
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.figsize'] = [15, 7]


# ==============================================================================
# Step 2: Load and Prepare Data
# ==============================================================================
try:
    df = pd.read_csv('usage_stats.csv')
    print("Successfully loaded 'usage_stats.csv'")
except FileNotFoundError:
    print("Could not find 'usage_stats.csv'. Using embedded sample data instead.")
    csv_data = '''timestamp,people_count,table_used,table_total,beanbag_used,beanbag_total,filename
2025-10-09 19:10:37,0,9,9,0,5,IMG_20251009_191037.jpg
2025-10-09 19:20:55,0,0,0,0,2,IMG_20251009_192055.jpg
2025-10-09 19:31:09,0,9,9,0,4,IMG_20251009_193109.jpg
2025-10-09 19:41:21,0,0,0,0,1,IMG_20251009_194121.jpg
2025-09-11 13:46:02,4,0,8,2,5,IMG_20250911_134602.jpg
2025-09-11 13:47:12,4,0,8,1,6,IMG_20250911_134712.jpg
2025-09-11 13:48:23,5,8,8,0,6,IMG_20250911_134823.jpg
2025-09-11 13:49:34,6,8,8,2,7,IMG_20250911_134934.jpg
2025-09-11 13:50:45,6,0,8,2,7,IMG_20250911_135045.jpg
2025-10-09 14:24:52,5,0,11,0,4,IMG_20251009_142452.jpg
2025-10-09 14:35:04,9,0,9,0,5,IMG_20251009_143504.jpg
2025-10-09 14:45:14,4,0,9,0,5,IMG_20251009_144514.jpg
2025-10-09 14:55:27,5,0,9,2,7,IMG_20251009_145527.jpg
2025-10-09 15:05:41,8,0,9,0,5,IMG_20251009_150541.jpg
2025-10-09 15:15:53,9,0,8,1,5,IMG_20251009_151553.jpg
2025-10-09 15:26:03,7,0,9,1,5,IMG_20251009_152603.jpg
2025-10-09 15:36:17,9,0,9,1,6,IMG_20251009_153617.jpg
'''
    df = pd.read_csv(io.StringIO(csv_data))

# --- Preprocessing ---
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)
df.sort_index(inplace=True)
df_resampled = df[['people_count', 'table_used', 'beanbag_used']].resample('H').mean()
df_resampled.ffill(inplace=True)
df_resampled.bfill(inplace=True)

# --- Train/Test Split ---
train_size = int(len(df_resampled) * 0.8)
train, test = df_resampled.iloc[:train_size], df_resampled.iloc[train_size:]
print(f"Train set size: {len(train)}")
print(f"Test set size: {len(test)}")


# ==============================================================================
# Step 3: Grid Search for Best SARIMAX Model
# ==============================================================================
print("\\n--- Starting Grid Search for best SARIMAX parameters ---")
p = d = q = range(0, 2)
pdq = list(itertools.product(p, d, q))
seasonal_pdq = [(x[0], x[1], x[2], 24) for x in pdq]
best_aic = np.inf
best_pdq = None
best_seasonal_pdq = None
best_model_fit = None
for param in pdq:
    for param_seasonal in seasonal_pdq:
        try:
            mod = SARIMAX(train['people_count'], exog=train[['table_used', 'beanbag_used']], order=param, seasonal_order=param_seasonal, enforce_stationarity=False, enforce_invertibility=False)
            results = mod.fit(disp=False)
            if results.aic < best_aic:
                best_aic = results.aic
                best_pdq = param
                best_seasonal_pdq = param_seasonal
                best_model_fit = results
        except:
            continue
print(f'--- Grid Search Finished ---\\n')
print(f'Best Parameters Found: SARIMAX{best_pdq}{best_seasonal_pdq}')


# ==============================================================================
# Step 4: Build and Compare Models
# ==============================================================================
print("\\n--- Building and Comparing Models ---")
target_col = 'people_count'
exog_cols = ['table_used', 'beanbag_used']
predictions = {}
# --- Model 1: ARIMA ---
predictions['ARIMA'] = ARIMA(train[target_col], order=(1, 1, 0)).fit().forecast(steps=len(test))
# --- Model 2: Exponential Smoothing ---
predictions['Exponential Smoothing'] = ExponentialSmoothing(train[target_col], trend='add', seasonal='add', seasonal_periods=24).fit().forecast(steps=len(test))
# --- Model 3: SARIMAX (Manual) ---
predictions['SARIMAX (Manual)'] = SARIMAX(train[target_col], exog=train[exog_cols], order=(1,1,0), seasonal_order=(1,0,0,24)).fit(disp=False).get_forecast(steps=len(test), exog=test[exog_cols]).predicted_mean
# --- Model 4: Best SARIMAX (GridSearch) ---
predictions['Best SARIMAX (GridSearch)'] = best_model_fit.get_forecast(steps=len(test), exog=test[exog_cols]).predicted_mean
print("All models trained successfully!")


# ==============================================================================
# Step 5: Evaluate Performance and Plot Comparisons
# ==============================================================================
performance_results = []
for name, pred in predictions.items():
    rmse = np.sqrt(mean_squared_error(test[target_col], pred))
    performance_results.append({'Model': name, 'RMSE': rmse})
performance_df = pd.DataFrame(performance_results).set_index('Model').sort_values(by='RMSE')
print("\\n--- Model Performance Comparison (RMSE) ---")
display(performance_df)

# --- Plot 1: Combined Comparison Graph ---
print("\\n--- Graph 1: Combined Model Comparison ---")
plt.figure(figsize=(16, 8))
plt.plot(train[target_col], label='Actual Data (Train)', color='black', alpha=0.4)
plt.plot(test[target_col], label='Actual Data (Test)', color='blue', linewidth=2)
for name, pred in predictions.items():
    plt.plot(pred.index, pred, label=f'{name} Forecast', linestyle='--')
plt.title('All Models vs. Actual Data', fontsize=16)
plt.xlabel('Timestamp')
plt.ylabel('People Count')
plt.legend()
plt.show()

# --- Plot 2: Individual Model Graphs ---
print("\\n--- Graph 2: Individual Model Comparisons ---")
fig, axes = plt.subplots(nrows=len(predictions), ncols=1, figsize=(16, 20), sharex=True)
fig.suptitle('Individual Model Performance vs. Actual Data', fontsize=20, y=0.93)

for i, (name, pred) in enumerate(predictions.items()):
    ax = axes[i]
    ax.plot(train[target_col], label='Actual Data (Train)', color='black', alpha=0.3)
    ax.plot(test[target_col], label='Actual Data (Test)', color='blue', linewidth=2)
    ax.plot(pred.index, pred, label=f'{name} Forecast', linestyle='--', color='red')
    ax.set_title(name, fontsize=14)
    ax.set_ylabel('People Count')
    ax.legend()
plt.xlabel('Timestamp')
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()


# ==============================================================================
# Step 6: Future Forecasting (Using the Best Model)
# ==============================================================================
print("\\n--- Forecasting Future Values ---")
# --- 6.1 Forecast for a specific date and time ---
target_date_str = '2025-10-14 14:00:00'
target_date = pd.to_datetime(target_date_str)
last_date = df_resampled.index[-1]
future_periods = (target_date.to_period('H') - last_date.to_period('H')).n + (7 * 24)
future_dates = pd.date_range(start=last_date + pd.Timedelta(hours=1), periods=future_periods, freq='H')
future_df = pd.DataFrame(index=future_dates, columns=exog_cols)
future_df['table_used'] = df_resampled['table_used'].mean()
future_df['beanbag_used'] = df_resampled['beanbag_used'].mean()
forecast_object = best_model_fit.get_forecast(steps=future_periods, exog=future_df)
forecast_df = forecast_object.conf_int()
forecast_df['predicted_mean'] = forecast_object.predicted_mean

try:
    specific_prediction = forecast_df.loc[target_date]['predicted_mean']
    print(f"\\n--- Specific Forecast for {target_date_str} ---")
    print(f'Predicted People Count: {specific_prediction:.2f}')
except KeyError:
    print(f"\\nCould not find a forecast for the specified date: {target_date_str}")

# --- 6.2 Plot the forecast for the next week ---
print("\\n--- Graph 3: Future Forecast for the Next 7 Days ---")
start_forecast_date = last_date + pd.Timedelta(days=1)
end_forecast_date = start_forecast_date + pd.Timedelta(days=7)
forecast_next_week = forecast_df.loc[start_forecast_date:end_forecast_date]

plt.figure(figsize=(16, 8))
plt.plot(df_resampled.index[-168:], df_resampled['people_count'][-168:], label='Actual Data (Last 7 Days)')
plt.plot(forecast_next_week.index, forecast_next_week['predicted_mean'], label='Forecast (Next 7 Days)', color='red', linestyle='--')
plt.fill_between(forecast_next_week.index, forecast_next_week.iloc[:, 0], forecast_next_week.iloc[:, 1], color='pink', alpha=0.3, label='95% Confidence Interval')
plt.title('Hourly People Count Forecast for the Next Week', fontsize=16)
plt.xlabel('Date and Time')
plt.ylabel('Predicted People Count')
plt.legend()
plt.grid(True)
plt.show()

# --- 6.3 Summarize forecast into a daily average table ---
daily_summary = forecast_next_week['predicted_mean'].resample('D').mean()
print("\\n--- Daily Average Forecast Summary for the Next Week ---")
display(daily_summary.to_frame(name='Predicted People Count (Average)'))

# ==============================================================================
# Step 7: Save All Results
# ==============================================================================
# --- Save performance metrics ---
performance_df.to_csv('model_performance.csv')
print("\\n✅ Model performance metrics saved to 'model_performance.csv'")

# --- Save combined data for dashboard ---
dashboard_df = df_resampled[[target_col]].copy()
dashboard_df.rename(columns={target_col: 'Actual_People_Count'}, inplace=True)
for name, pred in predictions.items():
    col_name = f"Forecast_{name.replace(' ', '_').replace('(', '').replace(')', '')}"
    dashboard_df[col_name] = pred
dashboard_df.to_csv('forecast_dashboard_data.csv')
print("✅ Combined data for dashboard saved to 'forecast_dashboard_data.csv'")

# --- Save the best model ---
best_model_filename = 'best_sarimax_model.pkl'
with open(best_model_filename, 'wb') as pkl_file:
    pickle.dump(best_model_fit, pkl_file)
print(f"✅ Best model ({performance_df.index[0]}) saved to '{best_model_filename}'")

--- 1. Preparing Data (Resampling to Hourly) ---
Successfully loaded 'usage_stats.csv'
Data has been resampled to Hourly. Total data points: 679
Train set: 543 points | Test set: 136 points
\n--- 2. Finding best SARIMAX model with auto_arima ---
